# Steam market analysis for Ubisoft: 00 Data exploration and cleaning
Inspects the raw nested JSON, checks data quality and validates the cleaning implemented in `setup_data`.

In [0]:
%run ./setup_data

# Steam analysis: shared data setup
Loads and cleans the Steam catalog and builds the dataframes used by every analysis notebook. It is executed by the other notebooks with `%run ./setup_data` and displays nothing on purpose (lighter published notebooks). The exploration of the raw data and the justification of each cleaning step are in `00_data_exploration`.

| Dataframe | Grain | Content |
|---|---|---|
| `games_df` | 1 row per game | cleaned columns (prices, dates, owners, reviews, ages, platforms) |
| `genres_df` | 1 row per (game, genre) | `genre` exploded |
| `languages_df` | 1 row per (game, language) | `language` exploded |
| `platforms_df` | 1 row per (game, platform) | Windows / Mac / Linux |
| `tags_df` | 1 row per (game, tag) | community tags and their vote count |

### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

### Raw schema
Each record has an `id` and a nested `data` struct. `tags` is a struct with one field per tag (hundreds of fields), so it is printed apart.

In [0]:
print(f"{raw_df.count():,} records")
raw_df.select("id", "data.*").drop("tags").printSchema()
print("Number of distinct tag fields:", len(raw_df.select("data.tags.*").columns))

55,691 records
root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- linux: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- ty

In [0]:
display(flat_df.drop("tags", "short_description", "header_image", "website").limit(10))

appid,categories,ccu,developer,discount,genre,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,type
10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,game
1000000,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud)",0,IndigoBlue Game Studio,0,"Action, Adventure, Indie",999,"English, Korean, Simplified Chinese",ASCENXION,5,"0 .. 20,000","List(false, false, true)",27,999,PsychoFlux Entertainment,2021/05/14,0,game
1000010,"List(Single-player, Partial Controller Support, Steam Achievements, Steam Cloud, Steam Trading Cards)",99,NEXT Studios,70,"Adventure, Indie, RPG, Strategy",1999,"Simplified Chinese, English, Japanese, Traditional Chinese, French, German, Spanish - Spain, Russian, Portuguese - Brazil",Crown Trick,646,"200,000 .. 500,000","List(false, false, true)",4032,599,"Team17, NEXT Studios",2020/10/16,0,game
1000030,"List(Multi-player, Single-player, Co-op, Steam Achievements, Steam Cloud, Shared/Split Screen, Full controller support, Steam Trading Cards, Shared/Split Screen Co-op, Remote Play on Phone, Remote Play on Tablet, Remote Play on TV, Remote Play Together)",76,Vertigo Gaming Inc.,0,"Action, Indie, Simulation, Strategy",1999,English,"Cook, Serve, Delicious! 3?!",115,"100,000 .. 200,000","List(false, true, true)",1575,1999,Vertigo Gaming Inc.,2020/10/14,0,game
1000040,List(Single-player),0,DoubleC Games,0,"Action, Casual, Indie, Simulation",199,Simplified Chinese,细胞战争,1,"0 .. 20,000","List(false, false, true)",0,199,DoubleC Games,2019/03/30,0,game
1000080,"List(Multi-player, Single-player, Steam Achievements, Full controller support, Steam Trading Cards)",3,IndieLeague Studio,60,"Action, Adventure, Indie, RPG",1999,"Simplified Chinese, English, Traditional Chinese, Japanese, Korean",Zengeon,462,"100,000 .. 200,000","List(false, true, true)",1018,799,2P Games,2019/06/24,0,game
1000100,"List(Single-player, Steam Achievements, Steam Cloud)",0,七月九日,0,"Adventure, Indie, RPG, Strategy",1299,"Japanese, Simplified Chinese, Traditional Chinese",干支セトラ 陽ノ卷｜干支etc. 陽之卷,6,"0 .. 20,000","List(false, false, true)",18,1299,Starship Studio,2019/01/24,0,game
1000110,"List(Multi-player, Single-player, Co-op, Online PvP, Online Co-op, PvP)",0,重庆环游者网络科技,0,"Action, Adventure, Casual, Free to Play, Massively Multiplayer",0,"English, Simplified Chinese, Traditional Chinese",Jumping Master(跳跳大咖),34,"20,000 .. 50,000","List(false, false, true)",50,0,重庆环游者网络科技,2019/04/8,0,game
1000130,"List(Single-player, Steam Achievements, Steam Leaderboards)",0,Simon Codrington,0,"Casual, Indie",299,English,Cube Defender,0,"0 .. 20,000","List(false, true, true)",6,299,Simon Codrington,2019/01/6,0,game
1000280,List(Single-player),0,Villain Role,0,"Indie, RPG",1399,"English, Simplified Chinese, Traditional Chinese",Tower of Origin2-Worm's Nest,12,"0 .. 20,000","List(false, false, true)",32,1399,Villain Role,2021/09/9,0,game


### Data quality checks

In [0]:
display(flat_df.groupBy("type").count().orderBy(F.desc("count")))

type,count
game,55690
hardware,1


In [0]:
checks = flat_df.select(
    F.count("*").alias("rows"),
    F.countDistinct("appid").alias("distinct_appid"),
    *[F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(f"empty_{c}")
      for c in ["name", "publisher", "developer", "genre", "languages", "price", "release_date", "required_age", "owners"]]
)
display(checks)

rows,distinct_appid,empty_name,empty_publisher,empty_developer,empty_genre,empty_languages,empty_price,empty_release_date,empty_required_age,empty_owners
55691,55691,0,134,127,161,11,0,99,0,0


In [0]:
# formats of release_date and required_age before parsing
display(flat_df.groupBy(F.length("release_date").alias("release_date_length"))
        .agg(F.count("*").alias("games"), F.min("release_date").alias("example_min"), F.max("release_date").alias("example_max"))
        .orderBy("release_date_length"))
display(flat_df.groupBy("required_age").count().orderBy(F.desc("count")))

release_date_length,games,example_min,example_max
0,99,,
7,123,2009/06,2022/04
9,15732,1998/11/8,2022/11/7
10,39737,1997/06/30,2022/11/11


required_age,count
0,55030
15,264
18,223
16,38
17,38
12,32
13,26
14,10
10,7
6,4


### Cleaning and feature engineering
- **prices** are strings in cents: converted to USD (`try_cast` returns null instead of failing on odd values, which matters on Spark 4 / serverless where ANSI mode is on).
- **release_date** comes as `yyyy/M/d` or only `yyyy/M`: parsed to a date, `release_year` and `release_month` extracted.
- **owners** is a range (`"20,000 .. 50,000"`): lower bound, upper bound and midpoint kept. It is an estimate of sales volume (SteamSpy).
- **reviews**: total, positive ratio and the Wilson lower bound of the positive ratio (ranks games by rating while penalising small review counts).
- **required_age**: first number extracted (`"MA 15+"` becomes 15).
- **estimated_revenue_usd** = owners midpoint x current price: a rough order of magnitude, not actual sales (prices change, discounts, regional pricing).
- Only `type == "game"` is kept; duplicated `appid` are removed.

In [0]:
print(f"{games_df.count():,} games after cleaning (type == 'game', deduplicated)")
display(games_df.select("name", "publisher", "price_usd", "discount_pct", "release_date", "release_date_clean", "owners",
                        "owners_mid", "total_reviews", "positive_ratio", "wilson_score", "required_age", "required_age_int",
                        "genres", "n_languages", "n_platforms").limit(10))

55,690 games after cleaning (type == 'game', deduplicated)


name,publisher,price_usd,discount_pct,release_date,release_date_clean,owners,owners_mid,total_reviews,positive_ratio,wilson_score,required_age,required_age_int,genres,n_languages,n_platforms
Drawngeon: Dungeons of Ink and Paper,DarkDes Labs,4.99,0,2019/05/24,2019-05-24,"20,000 .. 50,000",35000.0,68,0.8088235294117647,0.6998913522432758,0,0,"List(Adventure, Indie, RPG)",5,1
Gun Beat,Gun Beat,6.99,0,2019/01/14,2019-01-14,"0 .. 20,000",10000.0,11,0.7272727272727273,0.4343497069897538,0,0,"List(Action, Casual, Indie)",1,1
Glorious Companions,Ancient Forge,19.99,0,2019/06/4,2019-06-04,"0 .. 20,000",10000.0,105,0.6285714285714286,0.5331435691527343,0,0,"List(RPG, Strategy, Early Access)",3,1
Zibbs - Alien Survival,Mykhail Konokh,7.99,0,2020/11/8,2020-11-08,"0 .. 20,000",10000.0,21,0.5714285714285714,0.3654621629703245,0,0,"List(Action, Indie)",3,1
バグダス - デバッガー検定 -,Jiruo Software,0.99,0,2019/01/21,2019-01-21,"0 .. 20,000",10000.0,72,0.8333333333333334,0.730890790978952,0,0,"List(Casual, Indie)",1,1
Agartha,Fruitbat Factory,7.99,0,2019/03/20,2019-03-20,"0 .. 20,000",10000.0,10,0.7,0.3967732199795652,0,0,"List(Action, Indie)",2,1
Through the Darkest of Times,HandyGames,14.99,0,2020/01/30,2020-01-30,"50,000 .. 100,000",75000.0,881,0.8456299659477866,0.8202756753392,0,0,"List(Adventure, Indie, Simulation, Strategy)",7,2
她2 : 我还想再见到你 Her2 : I Want To See You Again,橘喵喵,3.99,0,2019/02/15,2019-02-15,"0 .. 20,000",10000.0,146,0.9178082191780822,0.8618333782636683,0,0,"List(Casual, Indie, Simulation)",2,1
Invasion Zero,NeatVibez,0.0,0,2019/01/11,2019-01-11,"0 .. 20,000",10000.0,33,0.45454545454545453,0.298426891258915,0,0,"List(Action, Free to Play, Indie, Massively Multiplayer)",7,3
Delicious - Emily's Road Trip,GameHouse,12.99,0,2019/06/4,2019-06-04,"0 .. 20,000",10000.0,105,0.8095238095238095,0.7240221679553136,0,0,"List(Adventure, Casual, Strategy)",16,2


In [0]:
# share of rows where parsing failed, to know the coverage of each indicator
display(games_df.select(
    F.round(F.avg(F.col("release_date_clean").isNull().cast("int")) * 100, 2).alias("pct_no_release_date"),
    F.round(F.avg(F.col("price_usd").isNull().cast("int")) * 100, 2).alias("pct_no_price"),
    F.round(F.avg(F.col("owners_mid").isNull().cast("int")) * 100, 2).alias("pct_no_owners"),
    F.round(F.avg((F.col("total_reviews") == 0).cast("int")) * 100, 2).alias("pct_no_reviews"),
    F.round(F.avg((F.col("n_genres") == 0).cast("int")) * 100, 2).alias("pct_no_genre"),
))

pct_no_release_date,pct_no_price,pct_no_owners,pct_no_reviews,pct_no_genre
0.18,0.0,0.0,0.29,0.29


In [0]:
for name, df in [("genres_df", genres_df), ("languages_df", languages_df), ("platforms_df", platforms_df), ("tags_df", tags_df)]:
    print(f"{name}: {df.count():,} rows")

genres_df: 157,110 rows
languages_df: 202,293 rows
platforms_df: 76,901 rows
tags_df: 675,477 rows
